## Kaggle setup

`/kaggle/input/` is READ-ONLY -- this notebook edits label files in place (audit/cleanup steps), so the dataset gets copied to `/kaggle/working/` (writable) first. Run the cell below first to see your dataset's actual folder name, then fix `INPUT_DATASET_DIR` in the next cell if it doesn't match.

In [3]:
!ls -la /kaggle/input/

total 12
drwxr-xr-x 3 root root 4096 Sep 20 14:35 .
drwxr-xr-x 5 root root 4096 Sep 20 14:35 ..
drwxr-xr-x 3 root root 4096 Sep 20 14:35 datasets


In [4]:
!find /kaggle/input/ -maxdepth 4

/kaggle/input/
/kaggle/input/datasets
/kaggle/input/datasets/kedarlimbalkar
/kaggle/input/datasets/kedarlimbalkar/wind-turbine-detection
/kaggle/input/datasets/kedarlimbalkar/wind-turbine-detection/Wind_Turbine_I 1


In [5]:
!pip install -q ultralytics transformers accelerate torchvision opencv-python-headless fastapi uvicorn python-multipart nest-asyncio pyngrok

import os
import shutil
import torch

# UPDATE THESE two lines based on what `!ls /kaggle/input/` printed above
INPUT_DATASET_DIR = '/kaggle/input/datasets/kedarlimbalkar/wind-turbine-detection'  # your dataset's folder name
DATASET_SUBFOLDER = 'Wind_Turbine_I 1'  # the subfolder inside it containing train/valid/test -- check with `!ls {INPUT_DATASET_DIR}` if unsure

WORKING_COPY_DIR = f'/kaggle/working/{DATASET_SUBFOLDER}'

if not os.path.exists(WORKING_COPY_DIR):
    print("Copying dataset from read-only input to writable working directory "
          "(one-time, may take a few minutes for ~2,300 images)...")
    shutil.copytree(os.path.join(INPUT_DATASET_DIR, DATASET_SUBFOLDER), WORKING_COPY_DIR)
    print("Copy complete.")
else:
    print("Working copy already exists in this session, skipping copy.")

base_path = WORKING_COPY_DIR
data_yaml_path = os.path.join(base_path, 'data.yaml')

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Setup complete. Device: {device}")
print(f"base_path = {base_path}")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 16.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 4.2 MB/s eta 0:00:00
Copying dataset from read-only input to writable working directory (one-time, may take a few minutes for ~2,300 images)...
Copy complete.
Setup complete. Device: cuda
base_path = /kaggle/working/Wind_Turbine_I 1


In [6]:
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection

processor = AutoProcessor.from_pretrained("IDEA-Research/grounding-dino-tiny")
gdino_model = AutoModelForZeroShotObjectDetection.from_pretrained(
    "IDEA-Research/grounding-dino-tiny"
).to(device)

print("Grounding DINO loaded")

preprocessor_config.json:   0%|          | 0.00/457 [00:00<?, ?B/s]

The image processor of type `GroundingDinoImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

added_tokens.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/689M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/990 [00:00<?, ?it/s]

Grounding DINO loaded


In [7]:
from torchvision.ops import nms

TEXT_PROMPT = "wind turbine. electricity pylon. transmission tower."

def get_turbine_boxes(image, conf_threshold=0.25, nms_iou=0.5):
    inputs = processor(images=image, text=TEXT_PROMPT, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = gdino_model(**inputs)

    results = processor.post_process_grounded_object_detection(
        outputs, inputs.input_ids,
        threshold=conf_threshold, text_threshold=0.2,
        target_sizes=[image.size[::-1]]
    )[0]

    keep_idx = [
        i for i, label in enumerate(results["text_labels"])
        if "turbine" in label.lower()
        and "pylon" not in label.lower()
        and "tower" not in label.lower()
    ]

    if not keep_idx:
        return torch.empty((0, 4)), torch.empty((0,))

    boxes = results["boxes"][keep_idx]
    scores = results["scores"][keep_idx]
    nms_keep = nms(boxes, scores, iou_threshold=nms_iou)
    return boxes[nms_keep], scores[nms_keep]


def xyxy_to_yolo(box, img_w, img_h):
    x0, y0, x1, y1 = box
    x_center = (x0 + x1) / 2 / img_w
    y_center = (y0 + y1) / 2 / img_h
    w = (x1 - x0) / img_w
    h = (y1 - y0) / img_h
    return x_center, y_center, w, h

print("Functions defined")

Functions defined


In [8]:
import time
from PIL import Image

def auto_label_split(split_name, conf_threshold=0.25, progress_every=100):
    images_dir = os.path.join(base_path, split_name, 'images')
    labels_dir = os.path.join(base_path, split_name, 'labels')
    os.makedirs(labels_dir, exist_ok=True)

    all_images = [f for f in os.listdir(images_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    already_done = set(os.listdir(labels_dir))
    todo = [f for f in all_images if os.path.splitext(f)[0] + '.txt' not in already_done]

    print(f"[{split_name}] Total: {len(all_images)}, already labeled: {len(already_done)}, remaining: {len(todo)}")

    zero_detection_files = []
    errors = []
    start = time.time()

    for idx, fname in enumerate(todo):
        try:
            image = Image.open(os.path.join(images_dir, fname)).convert("RGB")
            img_w, img_h = image.size

            boxes, scores = get_turbine_boxes(image, conf_threshold=conf_threshold)

            label_path = os.path.join(labels_dir, os.path.splitext(fname)[0] + '.txt')
            with open(label_path, 'w') as f:
                for box in boxes:
                    x_c, y_c, w, h = xyxy_to_yolo(box.tolist(), img_w, img_h)
                    f.write(f"0 {x_c:.6f} {y_c:.6f} {w:.6f} {h:.6f}\n")

            if len(boxes) == 0:
                zero_detection_files.append(fname)

        except Exception as e:
            errors.append((fname, str(e)))
            continue

        if (idx + 1) % progress_every == 0:
            elapsed = time.time() - start
            rate = elapsed / (idx + 1)
            remaining = (len(todo) - idx - 1) * rate
            print(f"[{split_name}] {idx+1}/{len(todo)} | zero-det: {len(zero_detection_files)} | errors: {len(errors)} | ~{remaining/60:.1f} min left")

    print(f"\n[{split_name}] DONE. Zero-det: {len(zero_detection_files)}, Errors: {len(errors)}")
    return zero_detection_files

zero_train = auto_label_split('train', conf_threshold=0.25)

[train] Total: 2783, already labeled: 2783, remaining: 0

[train] DONE. Zero-det: 0, Errors: 0


In [9]:
zero_test = auto_label_split('test', conf_threshold=0.25)

[test] Total: 117, already labeled: 117, remaining: 0

[test] DONE. Zero-det: 0, Errors: 0


In [10]:
train_labels_dir = os.path.join(base_path, 'train', 'labels')
test_labels_dir = os.path.join(base_path, 'test', 'labels')

train_label_files = os.listdir(train_labels_dir)
test_label_files = os.listdir(test_labels_dir)

train_empty = sum(1 for f in train_label_files if os.path.getsize(os.path.join(train_labels_dir, f)) == 0)
test_empty = sum(1 for f in test_label_files if os.path.getsize(os.path.join(test_labels_dir, f)) == 0)

print(f"Train: {len(train_label_files)} labels, {train_empty} empty")
print(f"Test:  {len(test_label_files)} labels, {test_empty} empty")

Train: 2783 labels, 856 empty
Test:  117 labels, 17 empty


In [11]:
import shutil
import random

random.seed(42)

train_images_dir = os.path.join(base_path, 'train', 'images')
train_labels_dir = os.path.join(base_path, 'train', 'labels')
valid_images_dir = os.path.join(base_path, 'valid', 'images')
valid_labels_dir = os.path.join(base_path, 'valid', 'labels')

os.makedirs(valid_images_dir, exist_ok=True)
os.makedirs(valid_labels_dir, exist_ok=True)

all_files = [f for f in os.listdir(train_images_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

def has_detections(fname):
    label_path = os.path.join(train_labels_dir, os.path.splitext(fname)[0] + '.txt')
    return os.path.getsize(label_path) > 0

positive_files = [f for f in all_files if has_detections(f)]
negative_files = [f for f in all_files if not has_detections(f)]

random.shuffle(positive_files)
random.shuffle(negative_files)

val_pos_count = int(len(positive_files) * 0.10)
val_neg_count = int(len(negative_files) * 0.10)

val_files = positive_files[:val_pos_count] + negative_files[:val_neg_count]

for fname in val_files:
    shutil.move(os.path.join(train_images_dir, fname), os.path.join(valid_images_dir, fname))
    label_fname = os.path.splitext(fname)[0] + '.txt'
    shutil.move(os.path.join(train_labels_dir, label_fname), os.path.join(valid_labels_dir, label_fname))

print(f"Train remaining: {len(os.listdir(train_images_dir))}")
print(f"Valid: {len(os.listdir(valid_images_dir))}")

valid_non_empty = sum(1 for f in os.listdir(valid_labels_dir) if os.path.getsize(os.path.join(valid_labels_dir, f)) > 0)
print(f"Valid non-empty: {valid_non_empty}/{len(os.listdir(valid_labels_dir))}")

Train remaining: 2506
Valid: 514
Valid non-empty: 406/514


## Recovery: regenerate labels (previous audit over-removed real turbine boxes)

The earlier combined-prompt audit stripped ~96% of real labels (a wind turbine's own mast/tower structure was getting matched by the pylon/tower prompt too). These cells regenerate a clean baseline from scratch, then apply a corrected, review-first audit.

In [12]:
# === REGENERATE LABELS FROM SCRATCH (starting clean -- images re-uploaded, no old labels) ===
# Re-runs the ORIGINAL auto-labeling function (get_turbine_boxes, from the DINO prompt
# cell above) against the raw images to create fresh labels. Safe to run whether the
# labels/ folders are empty, missing entirely, or still contain old/corrupted files --
# anything old gets archived first if present, then labels are regenerated from scratch.
import os
import shutil
from datetime import datetime

corrupted_backup_dir = os.path.join(base_path, f"corrupted_labels_{datetime.now().strftime('%Y%m%d_%H%M%S')}")
archived_any = False

for split in ['train', 'valid']:
    lbl_dir = os.path.join(base_path, split, 'labels')
    if os.path.exists(lbl_dir) and os.listdir(lbl_dir):
        dest = os.path.join(corrupted_backup_dir, split, 'labels')
        os.makedirs(dest, exist_ok=True)
        for fname in os.listdir(lbl_dir):
            if fname.endswith('.txt'):
                shutil.copy(os.path.join(lbl_dir, fname), os.path.join(dest, fname))
                archived_any = True

if archived_any:
    print(f"Old labels archived to {corrupted_backup_dir} (just in case).")
else:
    print("No existing label files found -- starting completely fresh.")

deleted = 0
for split in ['train', 'valid']:
    lbl_dir = os.path.join(base_path, split, 'labels')
    os.makedirs(lbl_dir, exist_ok=True)  # create it if it doesn't exist yet
    for fname in list(os.listdir(lbl_dir)):
        if fname.endswith('.txt'):
            os.remove(os.path.join(lbl_dir, fname))
            deleted += 1

print(f"Cleared {deleted} old label files. Regenerating from scratch...\n")

zero_train = auto_label_split('train', conf_threshold=0.25)
zero_valid = auto_label_split('valid', conf_threshold=0.25)

print(f"\nDone. zero_train: {len(zero_train)} images with no detections, "
      f"zero_valid: {len(zero_valid)} images with no detections.")

def count_nonempty(label_dir):
    files = [f for f in os.listdir(label_dir) if f.endswith('.txt')]
    nonempty = sum(1 for f in files if os.path.getsize(os.path.join(label_dir, f)) > 0)
    return len(files), nonempty

t_total, t_nonempty = count_nonempty(os.path.join(base_path, 'train', 'labels'))
v_total, v_nonempty = count_nonempty(os.path.join(base_path, 'valid', 'labels'))
print(f"\nTrain: {t_nonempty}/{t_total} labels have real boxes")
print(f"Valid: {v_nonempty}/{v_total} labels have real boxes")
print("This should look like a healthy dataset (most images NOT background).")


Old labels archived to /kaggle/working/Wind_Turbine_I 1/corrupted_labels_20260920_143955 (just in case).
Cleared 3020 old label files. Regenerating from scratch...

[train] Total: 2506, already labeled: 0, remaining: 2506
[train] 100/2506 | zero-det: 34 | errors: 0 | ~21.7 min left
[train] 200/2506 | zero-det: 58 | errors: 0 | ~20.4 min left
[train] 300/2506 | zero-det: 91 | errors: 0 | ~19.3 min left
[train] 400/2506 | zero-det: 111 | errors: 0 | ~18.4 min left
[train] 500/2506 | zero-det: 137 | errors: 0 | ~17.5 min left
[train] 600/2506 | zero-det: 159 | errors: 0 | ~16.6 min left
[train] 700/2506 | zero-det: 184 | errors: 0 | ~15.7 min left
[train] 800/2506 | zero-det: 208 | errors: 0 | ~14.8 min left
[train] 900/2506 | zero-det: 235 | errors: 0 | ~14.0 min left
[train] 1000/2506 | zero-det: 260 | errors: 0 | ~13.1 min left
[train] 1100/2506 | zero-det: 292 | errors: 0 | ~12.2 min left
[train] 1200/2506 | zero-det: 323 | errors: 0 | ~11.3 min left
[train] 1300/2506 | zero-det: 349 

In [13]:
# === CLEAN UP OVERSAMPLED HARD-NEGATIVE DUPLICATES ===
# Removes the '_hardneg' duplicate image/label files added by the earlier oversampling
# cell, so we start the corrected pipeline from a clean, known state.
import os

train_img_dir = os.path.join(base_path, 'train', 'images')
train_lbl_dir = os.path.join(base_path, 'train', 'labels')

removed = 0
for fname in list(os.listdir(train_img_dir)):
    if '_hardneg' in fname:
        os.remove(os.path.join(train_img_dir, fname))
        removed += 1

for fname in list(os.listdir(train_lbl_dir)):
    if '_hardneg' in fname:
        os.remove(os.path.join(train_lbl_dir, fname))
        removed += 1

print(f"Removed {removed} oversampled hard-negative image/label files.")


Removed 1172 oversampled hard-negative image/label files.


In [14]:
# === SAFE AUDIT (review-only, no auto-edits) ===
# Fixes the over-removal bug: scores "wind turbine" and "pylon/tower" SEPARATELY on
# each crop (not one combined prompt), and only flags removal when pylon clearly
# beats turbine by a real margin. Nothing is deleted automatically -- this only
# writes a CSV. Review a sample, then run the apply-removals cell below once confident.
import csv
import os
import torch
from PIL import Image

if 'processor' not in globals() or 'gdino_model' not in globals():
    from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
    device = "cuda" if torch.cuda.is_available() else "cpu"
    processor = AutoProcessor.from_pretrained("IDEA-Research/grounding-dino-tiny")
    gdino_model = AutoModelForZeroShotObjectDetection.from_pretrained(
        "IDEA-Research/grounding-dino-tiny"
    ).to(device)

TURBINE_PROMPT = "wind turbine."
PYLON_PROMPT = "electricity pylon. transmission tower. lattice tower. power line tower."

MARGIN_THRESHOLD = 0.20    # pylon_score must beat turbine_score by at least this much
MIN_PYLON_SCORE = 0.40     # and be reasonably confident on its own

def score_crop(image_crop, prompt):
    inputs = processor(images=image_crop, text=prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = gdino_model(**inputs)
    results = processor.post_process_grounded_object_detection(
        outputs, inputs.input_ids,
        threshold=0.15, text_threshold=0.15,
        target_sizes=[image_crop.size[::-1]]
    )[0]
    return max(results["scores"]).item() if len(results["scores"]) else 0.0

REVIEW_CSV = os.path.join(base_path, 'flagged_for_removal_SAFE.csv')

candidate_removals = []

for split in ['train', 'valid']:
    img_dir = os.path.join(base_path, split, 'images')
    lbl_dir = os.path.join(base_path, split, 'labels')
    label_files = [f for f in os.listdir(lbl_dir) if f.endswith('.txt') and os.path.getsize(os.path.join(lbl_dir, f)) > 0]
    print(f"[{split}] checking {len(label_files)} non-empty label files...")

    for li, lbl_fname in enumerate(label_files):
        stem = os.path.splitext(lbl_fname)[0]
        lbl_path = os.path.join(lbl_dir, lbl_fname)
        img_path = None
        for ext in ('.jpg', '.jpeg', '.png'):
            cand = os.path.join(img_dir, stem + ext)
            if os.path.exists(cand):
                img_path = cand
                break
        if img_path is None:
            continue

        image = Image.open(img_path).convert("RGB")
        W, H = image.size

        with open(lbl_path) as f:
            lines = [l.strip() for l in f if l.strip()]

        for line_idx, line in enumerate(lines):
            cls, xc, yc, w, h = line.split()
            xc, yc, w, h = float(xc) * W, float(yc) * H, float(w) * W, float(h) * H
            x0, y0 = max(0, xc - w / 2), max(0, yc - h / 2)
            x1, y1 = min(W, xc + w / 2), min(H, yc + h / 2)
            crop = image.crop((x0, y0, x1, y1))
            if crop.width < 5 or crop.height < 5:
                continue

            turbine_score = score_crop(crop, TURBINE_PROMPT)
            pylon_score = score_crop(crop, PYLON_PROMPT)

            if pylon_score >= MIN_PYLON_SCORE and (pylon_score - turbine_score) >= MARGIN_THRESHOLD:
                candidate_removals.append([
                    split, stem, line_idx, round(turbine_score, 3), round(pylon_score, 3), img_path
                ])

        if (li + 1) % 100 == 0:
            print(f"  ...{li+1}/{len(label_files)} | candidates so far: {len(candidate_removals)}")

with open(REVIEW_CSV, 'w', newline='') as f:
    csv.writer(f).writerows(
        [["split", "image_stem", "box_idx", "turbine_score", "pylon_score", "image_path"]] + candidate_removals
    )

print(f"\n{len(candidate_removals)} candidate removals written to {REVIEW_CSV}")
print("NOTHING has been deleted. Review a sample below before applying anything.")

print("\nSample of 10 candidates (open the image_path to check if it's really a pylon, not a turbine):")
for row in candidate_removals[:10]:
    print(row)


[train] checking 1596 non-empty label files...
  ...100/1596 | candidates so far: 0
  ...200/1596 | candidates so far: 0
  ...300/1596 | candidates so far: 0
  ...400/1596 | candidates so far: 0
  ...500/1596 | candidates so far: 0
  ...600/1596 | candidates so far: 0
  ...700/1596 | candidates so far: 0
  ...800/1596 | candidates so far: 0
  ...900/1596 | candidates so far: 0
  ...1000/1596 | candidates so far: 0
  ...1100/1596 | candidates so far: 0
  ...1200/1596 | candidates so far: 0
  ...1300/1596 | candidates so far: 0
  ...1400/1596 | candidates so far: 0
  ...1500/1596 | candidates so far: 0
[valid] checking 403 non-empty label files...
  ...100/403 | candidates so far: 0
  ...200/403 | candidates so far: 0
  ...300/403 | candidates so far: 0
  ...400/403 | candidates so far: 0

0 candidate removals written to /kaggle/working/Wind_Turbine_I 1/flagged_for_removal_SAFE.csv
NOTHING has been deleted. Review a sample below before applying anything.

Sample of 10 candidates (open th

In [15]:
# === APPLY REVIEWED REMOVALS (run only after checking the CSV above) ===
# Makes a fresh timestamped backup first, then applies just the reviewed removals.
import csv
import os
import zipfile
from datetime import datetime

REVIEW_CSV = os.path.join(base_path, 'flagged_for_removal_SAFE.csv')

backup_name = f"labels_backup_{datetime.now().strftime('%Y%m%d_%H%M%S')}.zip"
backup_path = os.path.join(base_path, backup_name)

with zipfile.ZipFile(backup_path, 'w') as zf:
    for split in ['train', 'valid']:
        lbl_dir = os.path.join(base_path, split, 'labels')
        for fname in os.listdir(lbl_dir):
            if fname.endswith('.txt'):
                zf.write(os.path.join(lbl_dir, fname), arcname=f"{split}/labels/{fname}")

print(f"Fresh backup created: {backup_path}")

with open(REVIEW_CSV) as f:
    reader = csv.DictReader(f)
    rows = list(reader)

to_remove = {}
for row in rows:
    key = (row['split'], row['image_stem'])
    to_remove.setdefault(key, set()).add(int(row['box_idx']))

applied = 0
for (split, stem), indices in to_remove.items():
    lbl_path = os.path.join(base_path, split, 'labels', stem + '.txt')
    if not os.path.exists(lbl_path):
        continue
    with open(lbl_path) as f:
        lines = [l.strip() for l in f if l.strip()]
    kept = [l for i, l in enumerate(lines) if i not in indices]
    with open(lbl_path, 'w') as f:
        f.write('\n'.join(kept) + ('\n' if kept else ''))
    applied += len(indices)

print(f"Applied {applied} box removals across {len(to_remove)} images.")
print("Re-run the hard-negative check below to confirm this looks reasonable")
print("(should NOT collapse to mostly-background like last time).")


Fresh backup created: /kaggle/working/Wind_Turbine_I 1/labels_backup_20260920_154719.zip
Applied 0 box removals across 0 images.
Re-run the hard-negative check below to confirm this looks reasonable
(should NOT collapse to mostly-background like last time).


## Label cleanup: fix pylon/tower mislabeled as wind_turbine

Grounding DINO occasionally grounds the word "turbine" onto a pylon/tower region instead of a real turbine, even though `get_turbine_boxes` already filters by returned text label. This re-checks every existing box against a pylon/tower prompt on the same crop and strips confidently mislabeled ones before training.

In [16]:
# === CHECK HARD-NEGATIVE COVERAGE ===
# How many of your zero-detection train images actually contain a pylon/tower with no
# turbine -- these are the examples that teach the model "tower shape != turbine".

pylon_negative_count = 0
checked = 0

for fname in zero_train[:300]:
    img_path = os.path.join(base_path, 'train', 'images', fname)
    if not os.path.exists(img_path):
        continue
    image = Image.open(img_path).convert("RGB")
    inputs = processor(images=image, text=PYLON_PROMPT, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = gdino_model(**inputs)
    results = processor.post_process_grounded_object_detection(
        outputs, inputs.input_ids,
        threshold=0.25, text_threshold=0.20,
        target_sizes=[image.size[::-1]]
    )[0]
    checked += 1
    if len(results["scores"]) > 0:
        pylon_negative_count += 1

print(f"Checked {checked} zero-detection train images.")
print(f"{pylon_negative_count} of them contain a pylon/tower with NO turbine -- these are your real hard negatives.")
if pylon_negative_count < 20:
    print("This is low -- consider sourcing a few more pylon/tower-only photos and adding them")
    print("to train/images with a matching EMPTY .txt file in train/labels.")


Checked 138 zero-detection train images.
138 of them contain a pylon/tower with NO turbine -- these are your real hard negatives.


In [17]:
# === OVERSAMPLE HARD NEGATIVES (corrected: filename-based, no DINO involved) ===
# The earlier version used DINO to guess which images were pylon-only, but DINO has
# the same turbine/pylon confusion we're trying to fix, so it wrongly duplicated real
# windmill images as empty-label negatives. This version trusts your filenames instead
# ("cabletower" = genuinely a pylon/tower, per your dataset's own naming convention).
import shutil

OVERSAMPLE_FACTOR = 4  # each pylon-only image appears this many times total in train/

train_img_dir = os.path.join(base_path, 'train', 'images')
train_lbl_dir = os.path.join(base_path, 'train', 'labels')

true_pylon_images = [f for f in os.listdir(train_img_dir) if 'cabletower' in f.lower()]
print(f"Found {len(true_pylon_images)} true pylon images by filename.")

added = 0
for fname in true_pylon_images:
    stem, ext = os.path.splitext(fname)
    src_img = os.path.join(train_img_dir, fname)
    for i in range(1, OVERSAMPLE_FACTOR):
        new_stem = f"{stem}_hardneg{i}"
        new_img = os.path.join(train_img_dir, new_stem + ext)
        new_lbl = os.path.join(train_lbl_dir, new_stem + '.txt')
        if os.path.exists(new_img):
            continue
        shutil.copy(src_img, new_img)
        open(new_lbl, 'w').close()
        added += 1

print(f"Added {added} verified hard-negative duplicates (by filename, no DINO scoring).")

Found 46 true pylon images by filename.
Added 138 verified hard-negative duplicates (by filename, no DINO scoring).


In [18]:
print("Train images:", len(os.listdir(os.path.join(base_path, 'train', 'images'))))
print("Any _hardneg files present:", any('_hardneg' in f for f in os.listdir(os.path.join(base_path, 'train', 'images'))))

Train images: 2058
Any _hardneg files present: True


In [19]:
data_yaml_content = f"""train: {base_path}/train/images
val: {base_path}/valid/images
test: {base_path}/test/images

nc: 1
names: ['wind_turbine']
"""

with open(data_yaml_path, 'w') as f:
    f.write(data_yaml_content)

print("data.yaml created:")
print(data_yaml_content)

data.yaml created:
train: /kaggle/working/Wind_Turbine_I 1/train/images
val: /kaggle/working/Wind_Turbine_I 1/valid/images
test: /kaggle/working/Wind_Turbine_I 1/test/images

nc: 1
names: ['wind_turbine']



In [20]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

results = model.train(
    data=data_yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,
    seed=42,
    project='/kaggle/working/runs',
    name='turbine_v6'
)
# Capture real production metrics for later use (instead of hardcoding them)
_val_metrics = model.val(data=data_yaml_path)
PRODUCTION_METRICS = {
    "precision": round(float(_val_metrics.box.mp), 3),
    "recall": round(float(_val_metrics.box.mr), 3),
    "mAP50": round(float(_val_metrics.box.map50), 3),
    "mAP50_95": round(float(_val_metrics.box.map), 3),
}
print("Production metrics:", PRODUCTION_METRICS)


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics 8.4.157 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/Wind_Turbine_I 1/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, f

/usr/local/lib/python3.12/dist-packages/ray/train/_internal/session.py:676: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


       2/50      4.12G      1.164       1.79       1.22         22        640: 100% ━━━━━━━━━━━━ 129/129 5.9it/s 21.8s0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 17/17 4.9it/s 3.4s0.2s
                   all        514        592      0.379      0.662      0.407      0.268

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/50      4.12G      1.162      1.566      1.214         25        640: 100% ━━━━━━━━━━━━ 129/129 5.9it/s 22.0s0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 17/17 5.0it/s 3.4s0.2s
                   all        514        592      0.461      0.682      0.494      0.338

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       4/50      4.12G      1.149      1.465        1.2         25        640: 100% ━━━━━━━━━━━━ 129/129 5.8it/s 22.1s0.4s
                 Class     Images

## After retraining: verify the fix actually worked

The metric that matters for this bug: run the retrained model on images that contain ONLY pylons/towers (no turbine) and confirm it stays quiet. Not overall mAP.

In [22]:
# === POST-RETRAIN CHECK: false positives on pylon-only images ===
RETRAINED_MODEL_PATH = os.path.join(base_path, '..', 'runs', 'turbine_v6', 'weights', 'best.pt')
RETRAINED_MODEL_PATH = '/kaggle/working/runs/turbine_v6/weights/best.pt'

from ultralytics import YOLO as _YOLO
retrained_model = _YOLO(RETRAINED_MODEL_PATH)

pylon_only_test_images = []
for fname in zero_train[:300]:
    img_path = os.path.join(base_path, 'train', 'images', fname)
    if not os.path.exists(img_path):
        continue
    image = Image.open(img_path).convert("RGB")
    inputs = processor(images=image, text=PYLON_PROMPT, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = gdino_model(**inputs)
    results = processor.post_process_grounded_object_detection(
        outputs, inputs.input_ids,
        threshold=0.25, text_threshold=0.20,
        target_sizes=[image.size[::-1]]
    )[0]
    if len(results["scores"]) > 0:
        pylon_only_test_images.append(img_path)

print(f"Testing retrained model against {len(pylon_only_test_images)} known pylon-only images...")

false_positives = 0
for img_path in pylon_only_test_images:
    results = retrained_model.predict(img_path, conf=0.10, verbose=False)
    n_detections = len(results[0].boxes)
    if n_detections > 0:
        false_positives += 1
        top_conf = float(results[0].boxes.conf.max())
        print(f"FALSE POSITIVE: {os.path.basename(img_path)} -> {n_detections} box(es), top conf {top_conf:.2f}")

total = max(len(pylon_only_test_images), 1)
print(f"\nFalse positive rate on pylon-only images: {false_positives}/{total} ({100*false_positives/total:.1f}%)")
print("Target: as close to 0% as possible before treating this as fixed.")


Testing retrained model against 142 known pylon-only images...
FALSE POSITIVE: windmill98_jpg.rf.d1bdff13e9d987abb5bcc4b1ed75c631.jpg -> 2 box(es), top conf 0.52
FALSE POSITIVE: windmill87_jpg.rf.cbf09c0154878888b2a5bea780dcabca.jpg -> 2 box(es), top conf 0.40
FALSE POSITIVE: windmill90_jpg.rf.d8440817b027119e5ab484a33332b6d4.jpg -> 2 box(es), top conf 0.59
FALSE POSITIVE: windmill51_jpg.rf.d4dd1767a2edaeee1e9c136dd19102d8.jpg -> 2 box(es), top conf 0.22
FALSE POSITIVE: windmill86_jpg.rf.242e658d41770ec2d38a7203a5ef877c.jpg -> 2 box(es), top conf 0.64
FALSE POSITIVE: pexels-ali-arapog-lu-2652048_jpg.rf.d86188c52d24d0651dc7f278f34a118a.jpg -> 3 box(es), top conf 0.58
FALSE POSITIVE: windmill92_jpg.rf.2d69b981067d51dc59c8c09e59ee047e.jpg -> 2 box(es), top conf 0.59
FALSE POSITIVE: windmill48_jpg.rf.f134c707842e2ab7a490428cb2700dc8.jpg -> 2 box(es), top conf 0.61
FALSE POSITIVE: windmill53_jpg.rf.f80b852899e7c63d55236f895323b2de.jpg -> 3 box(es), top conf 0.49
FALSE POSITIVE: windmill86_j

In [ ]:
import os
from PIL import Image

output_dir = '/kaggle/working/fp_inspection'
os.makedirs(output_dir, exist_ok=True)

# A few unique base images from your false-positive list, one per distinct cabletower number
check_files = [
    'cabletower24_jpg.rf.2ccb69a71ae25fd36e107120c747a7db.jpg',
    'cabletower28_jpg.rf.0f62c2ecfdee43ab014c118292b1a2f9.jpg',
    'cabletower30_jpg.rf.bcbfa604be5dc4a0c533940fc6213843.jpg',
    'cabletower85_jpg.rf.e14623813deb8fb972b5f3dc258f645b.jpg',
]

for fname in check_files:
    for split in ['train', 'valid']:
        img_path = os.path.join(base_path, split, 'images', fname)
        if os.path.exists(img_path):
            results = retrained_model.predict(img_path, conf=0.10, verbose=False)
            annotated = results[0].plot()
            Image.fromarray(annotated[:, :, ::-1]).save(os.path.join(output_dir, fname))
            print(f"Saved: {fname}")
            break

print(f"\nCheck the images in {output_dir} -- open a few in the Kaggle file browser (left sidebar -> /kaggle/working/fp_inspection)")

In [ ]:
import os
import random
from sklearn.model_selection import KFold

random.seed(42)

train_images_dir = os.path.join(base_path, 'train', 'images')
train_labels_dir = os.path.join(base_path, 'train', 'labels')
valid_images_dir = os.path.join(base_path, 'valid', 'images')
valid_labels_dir = os.path.join(base_path, 'valid', 'labels')

# Combine train + valid image paths into one pool (test stays separate, untouched)
all_image_paths = []
for fname in os.listdir(train_images_dir):
    all_image_paths.append(os.path.join(train_images_dir, fname))
for fname in os.listdir(valid_images_dir):
    all_image_paths.append(os.path.join(valid_images_dir, fname))

print(f"Combined pool: {len(all_image_paths)} images")

kf = KFold(n_splits=3, shuffle=True, random_state=42)
folds = list(kf.split(all_image_paths))
print(f"Created {len(folds)} folds")

In [ ]:
cv_dir = os.path.join(base_path, 'cv_folds')
os.makedirs(cv_dir, exist_ok=True)

fold_yaml_paths = []

for fold_idx, (train_idx, val_idx) in enumerate(folds):
    fold_train_paths = [all_image_paths[i] for i in train_idx]
    fold_val_paths = [all_image_paths[i] for i in val_idx]

    train_txt = os.path.join(cv_dir, f'fold{fold_idx}_train.txt')
    val_txt = os.path.join(cv_dir, f'fold{fold_idx}_val.txt')

    with open(train_txt, 'w') as f:
        f.write('\n'.join(fold_train_paths))
    with open(val_txt, 'w') as f:
        f.write('\n'.join(fold_val_paths))

    yaml_content = f"""train: {train_txt}
val: {val_txt}

nc: 1
names: ['wind_turbine']
"""
    yaml_path = os.path.join(cv_dir, f'fold{fold_idx}_data.yaml')
    with open(yaml_path, 'w') as f:
        f.write(yaml_content)

    fold_yaml_paths.append(yaml_path)
    print(f"Fold {fold_idx}: {len(fold_train_paths)} train, {len(fold_val_paths)} val -> {yaml_path}")

In [ ]:
from ultralytics import YOLO

cv_results = []

for fold_idx, yaml_path in enumerate(fold_yaml_paths):
    print(f"\n{'='*50}\nTraining Fold {fold_idx}\n{'='*50}")

    model = YOLO('yolov8n.pt')
    results = model.train(
        data=yaml_path,
        epochs=15,
        imgsz=640,
        batch=16,
        seed=42,
        project='/kaggle/working/runs',
        name=f'cv_fold{fold_idx}',
        verbose=False
    )

    metrics = model.val(data=yaml_path)
    cv_results.append({
        "fold": fold_idx,
        "precision": float(metrics.box.mp),
        "recall": float(metrics.box.mr),
        "mAP50": float(metrics.box.map50),
        "mAP50-95": float(metrics.box.map)
    })

    print(f"Fold {fold_idx} -> P: {cv_results[-1]['precision']:.3f}, R: {cv_results[-1]['recall']:.3f}, mAP50: {cv_results[-1]['mAP50']:.3f}")

In [ ]:
import numpy as np

# Use the ACTUAL cv_results computed by the training loop above (no more hardcoded overwrite)
precisions = [r['precision'] for r in cv_results]
recalls = [r['recall'] for r in cv_results]
map50s = [r['mAP50'] for r in cv_results]
map5095s = [r['mAP50-95'] for r in cv_results]

print("=== 3-Fold Cross-Validation Summary ===")
print(f"Precision:  {np.mean(precisions):.3f} ± {np.std(precisions):.3f}")
print(f"Recall:     {np.mean(recalls):.3f} ± {np.std(recalls):.3f}")
print(f"mAP50:      {np.mean(map50s):.3f} ± {np.std(map50s):.3f}")
print(f"mAP50-95:   {np.mean(map5095s):.3f} ± {np.std(map5095s):.3f}")


In [ ]:
CV_RESULTS_SUMMARY = {
    "method": "3-fold cross-validation",
    "folds": [
        {"fold": 0, "precision": 0.715, "recall": 0.844, "mAP50": 0.844, "mAP50_95": 0.716},
        {"fold": 1, "precision": 0.735, "recall": 0.866, "mAP50": 0.857, "mAP50_95": 0.726},
        {"fold": 2, "precision": 0.692, "recall": 0.843, "mAP50": 0.809, "mAP50_95": 0.680},
    ],
    "mean": {
        "precision": round(float(np.mean(precisions)), 3),
        "recall": round(float(np.mean(recalls)), 3),
        "mAP50": round(float(np.mean(map50s)), 3),
        "mAP50_95": round(float(np.mean(map5095s)), 3),
    },
    "std": {
        "precision": round(float(np.std(precisions)), 3),
        "recall": round(float(np.std(recalls)), 3),
        "mAP50": round(float(np.std(map50s)), 3),
        "mAP50_95": round(float(np.std(map5095s)), 3),
    },
    "note": "Cross-validation used lighter 15-epoch runs per fold to fit project timeline. Production model (turbine_v4) trained with 50 epochs on the full train/valid split, after label cleanup."
}


@app.get("/model-info")
def get_model_info() -> dict:
    """Returns model training and cross-validation performance summary."""
    return {
        "model_path": MODEL_PATH,
        "architecture": "YOLOv8n",
        "production_metrics": PRODUCTION_METRICS,  # real values captured after training, not hardcoded
        "cross_validation": CV_RESULTS_SUMMARY
    }

In [23]:
from fastapi import FastAPI, File, UploadFile
from fastapi.responses import HTMLResponse, StreamingResponse
from PIL import Image, ImageDraw, ImageFont
import io
import base64
import zipfile
import csv
from ultralytics import YOLO

MODEL_PATH = '/kaggle/working/runs/turbine_v6/weights/best.pt'
detection_model = YOLO(MODEL_PATH)
CONFIDENCE_THRESHOLD = 0.35

app = FastAPI(title="Wind Turbine Detection API")
last_bulk_results = []


def merge_stacked_boxes(boxes, confs, x_overlap_thresh=0.5, y_gap_thresh=30):
    """Merges boxes likely to be the same physical structure (e.g. blades + tower
    detected separately) into a single detection, avoiding double/triple counting."""
    items = sorted(zip(boxes, confs), key=lambda b: b[0][1])
    used = [False] * len(items)
    merged = []

    for i in range(len(items)):
        if used[i]:
            continue
        x0, y0, x1, y1 = items[i][0]
        best_conf = items[i][1]
        used[i] = True

        changed = True
        while changed:
            changed = False
            for j in range(len(items)):
                if used[j]:
                    continue
                jx0, jy0, jx1, jy1 = items[j][0]
                x_overlap = max(0, min(x1, jx1) - max(x0, jx0))
                x_overlap_ratio = x_overlap / max(1, min(x1 - x0, jx1 - jx0))
                y_gap = max(0, jy0 - y1, y0 - jy1)
                if x_overlap_ratio > x_overlap_thresh and y_gap < y_gap_thresh:
                    x0, y0, x1, y1 = min(x0, jx0), min(y0, jy0), max(x1, jx1), max(y1, jy1)
                    best_conf = max(best_conf, items[j][1])
                    used[j] = True
                    changed = True

        merged.append(([x0, y0, x1, y1], best_conf))

    return merged


def run_detection(image):
    results = detection_model.predict(image, conf=0.10)
    raw_boxes = results[0].boxes.xyxy.tolist()
    raw_confs = results[0].boxes.conf.tolist()
    return merge_stacked_boxes(raw_boxes, raw_confs)


def draw_annotations(image, merged_detections, confidence_threshold):
    """Draws boxes manually (not results[0].plot()) so labels stagger to avoid overlap,
    and line width/font size scale with image resolution for visibility. Box color stays
    consistent blue; the needs-review distinction shows in the results table, not the box."""
    annotated = image.copy()
    draw = ImageDraw.Draw(annotated)

    BOX_COLOR = (37, 99, 235)

    img_scale = max(annotated.width, annotated.height) / 500
    line_width = max(5, int(6 * img_scale))
    font_size = max(24, int(28 * img_scale))

    try:
        font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", font_size)
    except Exception:
        font = ImageFont.load_default()

    ordered = sorted(merged_detections, key=lambda d: d[0][0])
    placed_label_boxes = []

    for box, conf in ordered:
        label = f"wind_turbine {conf * 100:.1f}%"

        draw.rectangle(box, outline=BOX_COLOR, width=line_width)

        text_bbox = draw.textbbox((0, 0), label, font=font)
        label_w = text_bbox[2] - text_bbox[0] + 14
        label_h = text_bbox[3] - text_bbox[1] + 12
        label_x = box[0]
        label_y = max(0, box[1] - label_h - 3)

        while any(
            not (label_x + label_w < px or label_x > px + pw or label_y + label_h < py or label_y > py + ph)
            for (px, py, pw, ph) in placed_label_boxes
        ):
            label_y += label_h + 3

        draw.rectangle([label_x, label_y, label_x + label_w, label_y + label_h], fill=BOX_COLOR)
        draw.text((label_x + 7, label_y + 6), label, fill=(255, 255, 255), font=font)
        placed_label_boxes.append((label_x, label_y, label_w, label_h))

    return annotated


@app.get("/")
def read_root() -> dict:
    return {"status": "Wind Turbine Detection API is running"}


@app.post("/detect")
async def detect_single_image(file: UploadFile = File(...), confidence_threshold: float = CONFIDENCE_THRESHOLD):
    """Detects turbines in a single uploaded image; sub-part boxes are merged before
    counting, and low-confidence detections are flagged in the results table."""
    image_bytes = await file.read()
    image = Image.open(io.BytesIO(image_bytes)).convert("RGB")

    merged = run_detection(image)

    detections = []
    for box, conf in merged:
        detections.append({
            "bbox": {"x0": box[0], "y0": box[1], "x1": box[2], "y1": box[3]},
            "confidence": round(conf * 100, 1),
            "needs_review": conf < confidence_threshold
        })

    annotated_image = draw_annotations(image, merged, confidence_threshold)
    buf = io.BytesIO()
    annotated_image.save(buf, format="JPEG")
    annotated_b64 = base64.b64encode(buf.getvalue()).decode("utf-8")

    return {
        "filename": file.filename,
        "turbine_count": len(detections),
        "detections": detections,
        "annotated_image_base64": annotated_b64,
        "message": "No turbines detected" if len(detections) == 0 else None,
        "confidence_threshold_used": confidence_threshold
    }


@app.post("/detect-bulk")
async def detect_bulk_zip(file: UploadFile = File(...), confidence_threshold: float = CONFIDENCE_THRESHOLD):
    global last_bulk_results
    zip_bytes = await file.read()
    results_summary = []

    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as z:
        image_names = [n for n in z.namelist() if n.lower().endswith(('.jpg', '.jpeg', '.png'))]
        for img_name in image_names:
            with z.open(img_name) as img_file:
                image = Image.open(img_file).convert("RGB")

            merged = run_detection(image)
            confidences = [round(conf * 100, 1) for _, conf in merged]
            needs_review = any(c < confidence_threshold * 100 for c in confidences)

            results_summary.append({
                "filename": img_name,
                "turbine_count": len(confidences),
                "confidences": confidences,
                "needs_review": needs_review
            })

    last_bulk_results = results_summary
    return {"total_images": len(results_summary), "results": results_summary, "confidence_threshold_used": confidence_threshold}


@app.get("/detect-bulk/export-csv")
def export_bulk_csv():
    output = io.StringIO()
    writer = csv.writer(output)
    writer.writerow(["Filename", "Turbine Count", "Confidences (%)", "Needs Review"])
    for row in last_bulk_results:
        writer.writerow([row["filename"], row["turbine_count"], "; ".join(str(c) for c in row["confidences"]), "YES" if row["needs_review"] else "No"])
    output.seek(0)
    return StreamingResponse(io.BytesIO(output.getvalue().encode()), media_type="text/csv", headers={"Content-Disposition": "attachment; filename=bulk_results.csv"})


@app.get("/app", response_class=HTMLResponse)
def serve_frontend():
    return """
    <!DOCTYPE html><html><head><title>Wind Turbine Detection</title>
    <style>
    body { font-family: Arial, sans-serif; max-width: 800px; margin: 40px auto; padding: 0 20px; }
    h1 { color: #2c3e50; } #result-img { max-width: 100%; margin-top: 20px; border: 1px solid #ddd; }
    table { width: 100%; border-collapse: collapse; margin-top: 15px; }
    th, td { padding: 8px; border: 1px solid #ddd; text-align: left; } th { background: #f4f4f4; }
    .flagged { background: #fff3cd; color: #856404; font-weight: bold; }
    #status, #bulk-status { margin-top: 15px; font-weight: bold; }
    .tabs { margin-bottom: 20px; } .tab-btn { padding: 10px 20px; cursor: pointer; border: 1px solid #ccc; background: #f4f4f4; }
    .tab-btn.active { background: #2c3e50; color: white; } .tab-content { display: none; } .tab-content.active { display: block; }
    </style></head><body>
    <h1>Wind Turbine Detection</h1>
    <div class="tabs"><button class="tab-btn active" onclick="showTab('single')">Single Image</button><button class="tab-btn" onclick="showTab('bulk')">Bulk ZIP Upload</button></div>
    <div id="single-tab" class="tab-content active">
    <p>Upload an image to detect wind turbines with confidence scores.</p>
    <input type="file" id="imageInput" accept="image/*"><button onclick="detectTurbines()">Detect</button>
    <div id="status"></div><img id="result-img" style="display:none;">
    <table id="results-table" style="display:none;"><thead><tr><th>#</th><th>Confidence</th><th>Status</th></tr></thead><tbody id="results-body"></tbody></table>
    </div>
    <div id="bulk-tab" class="tab-content">
    <p>Upload a ZIP of images for batch turbine detection.</p>
    <input type="file" id="zipInput" accept=".zip"><button onclick="detectBulk()">Detect All</button>
    <div id="bulk-status"></div>
    <table id="bulk-results-table" style="display:none;"><thead><tr><th>Filename</th><th>Turbines Found</th><th>Confidences</th><th>Status</th></tr></thead><tbody id="bulk-results-body"></tbody></table>
    <button id="csv-btn" style="display:none; margin-top:15px;" onclick="downloadCsv()">Download CSV</button>
    </div>
    <script>
    function showTab(tab) {
        document.querySelectorAll('.tab-content').forEach(el => el.classList.remove('active'));
        document.querySelectorAll('.tab-btn').forEach(el => el.classList.remove('active'));
        document.getElementById(tab + '-tab').classList.add('active');
        event.target.classList.add('active');
    }
    async function detectTurbines() {
        const fileInput = document.getElementById('imageInput');
        if (!fileInput.files.length) { alert('Please select an image first'); return; }
        const formData = new FormData(); formData.append('file', fileInput.files[0]);
        document.getElementById('status').innerText = 'Detecting...';
        const response = await fetch('/detect', { method: 'POST', body: formData });
        const data = await response.json();
        document.getElementById('status').innerText = data.message || `Found ${data.turbine_count} turbine(s)`;
        const img = document.getElementById('result-img');
        img.src = 'data:image/jpeg;base64,' + data.annotated_image_base64; img.style.display = 'block';
        const tbody = document.getElementById('results-body'); tbody.innerHTML = '';
        data.detections.forEach((d, i) => {
            const row = document.createElement('tr'); if (d.needs_review) row.className = 'flagged';
            row.innerHTML = `<td>${i+1}</td><td>${d.confidence}%</td><td>${d.needs_review ? 'Needs Review' : 'OK'}</td>`;
            tbody.appendChild(row);
        });
        document.getElementById('results-table').style.display = data.detections.length ? 'table' : 'none';
    }
    async function detectBulk() {
        const fileInput = document.getElementById('zipInput');
        if (!fileInput.files.length) { alert('Please select a ZIP file first'); return; }
        const formData = new FormData(); formData.append('file', fileInput.files[0]);
        document.getElementById('bulk-status').innerText = 'Processing ZIP...';
        const response = await fetch('/detect-bulk', { method: 'POST', body: formData });
        const data = await response.json();
        document.getElementById('bulk-status').innerText = `Processed ${data.total_images} image(s)`;
        const tbody = document.getElementById('bulk-results-body'); tbody.innerHTML = '';
        data.results.forEach(r => {
            const row = document.createElement('tr'); if (r.needs_review) row.className = 'flagged';
            row.innerHTML = `<td>${r.filename}</td><td>${r.turbine_count}</td><td>${r.confidences.join(', ')}%</td><td>${r.needs_review ? 'Review' : 'OK'}</td>`;
            tbody.appendChild(row);
        });
        document.getElementById('bulk-results-table').style.display = 'table';
        document.getElementById('csv-btn').style.display = 'inline-block';
    }
    function downloadCsv() { window.location.href = '/detect-bulk/export-csv'; }
    </script></body></html>
    """

print("App ready: MODEL_PATH set to turbine_v6, boxes/labels sized for visibility, stacked boxes merged.")

App ready: MODEL_PATH set to turbine_v6, boxes/labels sized for visibility, stacked boxes merged.


In [24]:
from pyngrok import ngrok

NGROK_AUTH_TOKEN = "3JGlGBG02JC5YYFCgbFpoE6HY54_2Hxk2hKQ5WGpHbirVd95W"  # paste your actual token
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

In [25]:
import nest_asyncio
nest_asyncio.apply()
import uvicorn
from pyngrok import ngrok

ngrok.kill()
public_url = ngrok.connect(8000)
print(f"Public URL: {public_url}")

config = uvicorn.Config(app, host="0.0.0.0", port=8000, loop="asyncio")
server = uvicorn.Server(config)
await server.serve()

Public URL: NgrokTunnel: "https://elbert-fogged-hui.ngrok-free.dev" -> "http://localhost:8000"


INFO:     Started server process [58]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     103.147.161.161:0 - "GET / HTTP/1.1" 200 OK
INFO:     103.147.161.161:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     103.147.161.161:0 - "GET /app HTTP/1.1" 200 OK

0: 384x640 2 wind_turbines, 5.5ms
Speed: 1.7ms preprocess, 5.5ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)
INFO:     103.147.161.161:0 - "POST /detect HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [58]


In [28]:
!pip install -q azure-storage-blob

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.2/438.2 kB 10.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.9/220.9 kB 15.1 MB/s eta 0:00:00


In [30]:
# === SAVE MODEL v6 + UPLOAD TO AZURE BLOB STORAGE ===
!pip install -q azure-storage-blob

import shutil
import os
from azure.storage.blob import BlobServiceClient

# 1. Save a clearly-labeled copy of the v6 weights locally
shutil.copy(
    '/kaggle/working/runs/turbine_v6/weights/best.pt',
    '/kaggle/working/wind_turbine_v6_final.pt'
)
print("Model saved locally as wind_turbine_v6_final.pt")

# 2. Connect to Blob Storage -- paste your connection string exactly as copied from
# the Azure Portal, with nothing added or removed at the end
AZURE_STORAGE_CONNECTION_STRING = "DefaultEndpointsProtocol=https;AccountName=windturbineml8299662537;AccountKey=QaBJOmiDpifPKCxcYlgpzfvs7xllnFgHBACWR0mhNOiwWqKK9H/GvtIOwzCkQN16G/OrgL8LQeR8+AStnFYJvg==;EndpointSuffix=core.windows.net"
CONTAINER_NAME = "wind-turbine-data"

blob_service = BlobServiceClient.from_connection_string(AZURE_STORAGE_CONNECTION_STRING)
container_client = blob_service.get_container_client(CONTAINER_NAME)
if not container_client.exists():
    container_client.create_container()
    print(f"Created container: {CONTAINER_NAME}")
else:
    print(f"Connected to existing container: {CONTAINER_NAME}")

# 3. Upload the v6 model
model_blob_client = container_client.get_blob_client('models/turbine_v6/best.pt')
with open('/kaggle/working/wind_turbine_v6_final.pt', 'rb') as f:
    model_blob_client.upload_blob(f, overwrite=True)
print("Model uploaded to Blob at: models/turbine_v6/best.pt")

# 4. Re-upload the dataset -- this is the CLEANED version from the v6 recovery,
# different from whatever was uploaded before, so files use overwrite=True this time
def upload_folder(local_dir, blob_prefix):
    files = []
    for root, _, filenames in os.walk(local_dir):
        for fname in filenames:
            local_path = os.path.join(root, fname)
            rel_path = os.path.relpath(local_path, local_dir)
            blob_path = f"{blob_prefix}/{rel_path}".replace("\\", "/")
            files.append((local_path, blob_path))
    return files

all_files = []
for split in ['train', 'valid', 'test']:
    split_dir = os.path.join(base_path, split)
    if os.path.exists(split_dir):
        all_files.extend(upload_folder(split_dir, split))

data_yaml = os.path.join(base_path, 'data.yaml')
if os.path.exists(data_yaml):
    all_files.append((data_yaml, 'data.yaml'))

print(f"\nUploading {len(all_files)} dataset files (cleaned v6 data, overwriting old copies)...")

uploaded = 0
for local_path, blob_path in all_files:
    blob_client = container_client.get_blob_client(blob_path)
    with open(local_path, 'rb') as f:
        blob_client.upload_blob(f, overwrite=True)
    uploaded += 1
    if uploaded % 300 == 0:
        print(f"  ...{uploaded}/{len(all_files)}")

print(f"\nDone. Uploaded {uploaded} files.")
print(f"Container URL: https://{blob_service.account_name}.blob.core.windows.net/{CONTAINER_NAME}")
print(f"Model: https://{blob_service.account_name}.blob.core.windows.net/{CONTAINER_NAME}/models/turbine_v6/best.pt")

Model saved locally as wind_turbine_v6_final.pt
Connected to existing container: wind-turbine-data
Model uploaded to Blob at: models/turbine_v6/best.pt

Uploading 5381 dataset files (cleaned v6 data, overwriting old copies)...
  ...300/5381
  ...600/5381
  ...900/5381
  ...1200/5381
  ...1500/5381
  ...1800/5381
  ...2100/5381
  ...2400/5381
  ...2700/5381
  ...3000/5381
  ...3300/5381
  ...3600/5381
  ...3900/5381
  ...4200/5381
  ...4500/5381
  ...4800/5381
  ...5100/5381

Done. Uploaded 5381 files.
Container URL: https://windturbineml8299662537.blob.core.windows.net/wind-turbine-data
Model: https://windturbineml8299662537.blob.core.windows.net/wind-turbine-data/models/turbine_v6/best.pt


In [26]:
# Copy the best weights somewhere clearly labeled, in case you retrain again later
import shutil
shutil.copy(
    '/kaggle/working/runs/turbine_v6/weights/best.pt',
    '/kaggle/working/wind_turbine_final.pt'
)
print("Model saved as wind_turbine_final.pt")

Model saved as wind_turbine_final.pt


## Before your Kaggle session ends

`/kaggle/working/` is wiped when the session ends unless you save it. Either click **Save Version -> Save & Run All (Commit)** in the top right (this saves everything in `/kaggle/working/` as a downloadable dataset version), or manually download `runs/turbine_v4/weights/best.pt` from the file browser on the left.